# Setup environment (one-time, by hand)
- Creates, for one environment of this company: **S3 folders**, **catalog**, **schemas** and **volumes**
- (bronze volumes external on S3, ops volumes managed). Catalog is derived as `<company>_<env>`.
- **Widgets:** `all` ticked = whole list is used (even if other items also show ticked); untick `all` and tick items = only those.
- Cell 3 prints the **resolved selection**, which is exactly what will be used.
- **How:** Environment → Mode = preview → Run all → check → Mode = apply → Run all.
- Needs the external location `lh_<company>_<env>_source` (created by an admin). Safe to re-run. Not part of any job.

## 2. Repo constant and widgets

In [0]:
# The ONLY two lines that differ between repos
REPO_COMPANY = "fqf"
COMPANY_NAME = "Flexi Quant Finance"

S3_FOLDERS   = ["landing", "processed", "rejected"]
SCHEMA_NAMES = ["bronze", "silver", "gold", "ops"]
VOLUME_NAMES = ["landing", "processed", "rejected", "checkpoints", "refresh_in", "refresh_out"]

dbutils.widgets.dropdown("env",  "dev",     ["dev", "test", "prod"], "Environment")
dbutils.widgets.dropdown("mode", "preview", ["preview", "apply"],    "Mode")
dbutils.widgets.multiselect("s3_folders", "all", ["all"] + S3_FOLDERS,   "S3 folders")
dbutils.widgets.multiselect("schemas",    "all", ["all"] + SCHEMA_NAMES, "Schemas")
dbutils.widgets.multiselect("volumes",    "all", ["all"] + VOLUME_NAMES, "Volumes")

## 3. Configuration (all names derived; nothing typed by hand)

In [0]:
env  = dbutils.widgets.get("env")
mode = dbutils.widgets.get("mode")

def pick(name, full):
    """'all' ticked -> whole list; otherwise only the ticked items (in list order)."""
    vals = {v for v in dbutils.widgets.get(name).split(",") if v}
    return list(full) if "all" in vals else [v for v in full if v in vals]

catalog = f"{REPO_COMPANY}_{env}"                 # fqf_dev
bucket  = f"lh-{REPO_COMPANY}-{env}-source"       # lh-fqf-dev-source
ext_loc = f"lh_{REPO_COMPANY}_{env}_source"       # lh_fqf_dev_source

SCHEMA_COMMENTS = {"bronze": "Raw data, as landed", "silver": "Cleaned and conformed data",
                   "gold": "Business-ready tables", "ops": "Audit, watermarks and checks"}
VOLUME_RULES = {   # volume: (schema, type, comment, allowed envs)
    "landing":     ("bronze", "external", "New source files: <source>/<table>/yyyy/mm/", {"dev", "test", "prod"}),
    "processed":   ("bronze", "external", "Files loaded into bronze successfully",      {"dev", "test", "prod"}),
    "rejected":    ("bronze", "external", "Files that failed checks",                   {"dev", "test", "prod"}),
    "checkpoints": ("ops",    "managed",  "Auto Loader progress",                       {"dev", "test", "prod"}),
    "refresh_in":  ("ops",    "managed",  "Masked refresh files received",              {"dev", "test"}),
    "refresh_out": ("ops",    "managed",  "Masked refresh files waiting to be sent",    {"test", "prod"}),
}

sel_folders = pick("s3_folders", S3_FOLDERS)
sel_schemas = pick("schemas", SCHEMA_NAMES)
all_vols    = pick("volumes", VOLUME_NAMES)
sel_volumes = [v for v in all_vols if env in VOLUME_RULES[v][3]]
skipped     = [v for v in all_vols if env not in VOLUME_RULES[v][3]]

print(f"{COMPANY_NAME} | catalog={catalog} | bucket={bucket} | mode={mode}")
print("Resolved selection (this is exactly what will be used):")
print(f"  S3 folders : {sel_folders or 'none'}")
print(f"  Schemas    : {sel_schemas or 'none'}")
print(f"  Volumes    : {sel_volumes or 'none'}")
if skipped:
    print(f"  Not used in {env}, skipped: {skipped}")

## 4. Pre-check that the external location exists and points at the right bucket

In [0]:
needs_s3 = sel_folders or any(VOLUME_RULES[v][1] == "external" for v in sel_volumes)
if needs_s3:
    try:
        loc = spark.sql(f"DESCRIBE EXTERNAL LOCATION `{ext_loc}`").collect()[0].asDict()
    except Exception as e:
        raise Exception(f"External location '{ext_loc}' not found. Ask the admin to create it for {catalog} first.") from e
    if loc["url"].rstrip("/") != f"s3://{bucket}":
        raise Exception(f"'{ext_loc}' points to {loc['url']}, expected s3://{bucket}/")
    print(f"OK: {ext_loc} -> {loc['url']} (credential {loc['credential_name']})")
else:
    print("External location not needed for this selection.")

## 5. Build the SQL

In [0]:
folders = [f"s3://{bucket}/{f}/" for f in sel_folders]
stmts = []
if sel_schemas or sel_volumes:
    stmts.append(f"CREATE CATALOG IF NOT EXISTS {catalog} COMMENT '{COMPANY_NAME}, {env} environment'")
stmts += [f"CREATE SCHEMA IF NOT EXISTS {catalog}.{s} COMMENT '{SCHEMA_COMMENTS[s]}'" for s in sel_schemas]
for v in sel_volumes:
    schema, vtype, comment, _ = VOLUME_RULES[v]
    if vtype == "external":
        stmts.append(f"CREATE EXTERNAL VOLUME IF NOT EXISTS {catalog}.{schema}.{v} "
                     f"LOCATION 's3://{bucket}/{v}/' COMMENT '{comment}'")
    else:
        stmts.append(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{v} COMMENT '{comment}'")

for f in folders:
    print(f"MKDIR {f}")
for s in stmts:
    print(s + ";")
print(f"\nPlan: {len(folders)} folder(s), {len(stmts)} SQL statement(s)")

## 6. Apply (only when Mode = apply)

In [0]:
if mode == "apply":
    for f in folders:
        dbutils.fs.mkdirs(f)
    for s in stmts:
        spark.sql(s)
    print(f"Applied {len(folders)} folder(s) and {len(stmts)} statement(s) to {catalog}")
else:
    print("Preview only. Set Mode = apply and Run all to create.")

## 7. Verify, and flag anything missing or extra

In [0]:
if sel_folders:
    found = {f.name.rstrip("/") for f in dbutils.fs.ls(f"s3://{bucket}/")}
    for f in S3_FOLDERS:
        print(f"S3 folder {f}/: {'OK' if f in found else 'MISSING'}")

df = spark.sql(f"""
  SELECT volume_schema, volume_name, volume_type, storage_location
  FROM system.information_schema.volumes
  WHERE volume_catalog = '{catalog}'
  ORDER BY volume_schema, volume_name""")
display(df)
expected = {(VOLUME_RULES[v][0], v) for v in VOLUME_RULES if env in VOLUME_RULES[v][3]}
actual   = {(r.volume_schema, r.volume_name) for r in df.collect()}
print("Missing:", sorted(expected - actual) or "none")
print("Extra (overhead):", sorted(actual - expected) or "none")

## 8. Schemas

In [0]:
if spark.sql(f"SHOW CATALOGS LIKE '{catalog}'").count() == 1:
    display(spark.sql(f"SHOW SCHEMAS IN {catalog}"))
else:
    print(f"Catalog {catalog} does not exist yet.")